## Loading the IMDB dataset

In [5]:
vocab_size=10000

from tensorflow.keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=vocab_size)

In [ ]:
for doc_index in range(0,10):
  print(f'[{doc_index}] \t label:{train_labels[doc_index]} \tdoc length:{str(len(train_data[doc_index])).ljust(4)}   doc[0:20]:{train_data[doc_index][0:20]}')

[0] 	 label:1 	doc length:218    doc[0:20]:[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
[1] 	 label:0 	doc length:189    doc[0:20]:[1, 194, 1153, 194, 8255, 78, 228, 5, 6, 1463, 4369, 5012, 134, 26, 4, 715, 8, 118, 1634, 14]
[2] 	 label:0 	doc length:141    doc[0:20]:[1, 14, 47, 8, 30, 31, 7, 4, 249, 108, 7, 4, 5974, 54, 61, 369, 13, 71, 149, 14]
[3] 	 label:1 	doc length:550    doc[0:20]:[1, 4, 2, 2, 33, 2804, 4, 2040, 432, 111, 153, 103, 4, 1494, 13, 70, 131, 67, 11, 61]
[4] 	 label:0 	doc length:147    doc[0:20]:[1, 249, 1323, 7, 61, 113, 10, 10, 13, 1637, 14, 20, 56, 33, 2401, 18, 457, 88, 13, 2626]
[5] 	 label:0 	doc length:43     doc[0:20]:[1, 778, 128, 74, 12, 630, 163, 15, 4, 1766, 7982, 1051, 2, 32, 85, 156, 45, 40, 148, 139]
[6] 	 label:1 	doc length:123    doc[0:20]:[1, 6740, 365, 1234, 5, 1156, 354, 11, 14, 5327, 6638, 7, 1016, 2, 5940, 356, 44, 4, 1349, 500]
[7] 	 label:0 	doc length:562    doc[0:20]:[1, 4, 2, 716, 4, 65, 7, 4,

In [ ]:
max([max(sequence) for sequence in train_data])

9999

## Decoding reviews back to text

In [2]:
word_index = imdb.get_word_index()
reverse_word_index = dict(
    [(value, key) for (key, value) in word_index.items()])


1641221/1641221 [==============================] - 0s 0us/step


In [ ]:
input_word = "amazing"
index = word_index.get(input_word)

if index is not None:
    print(f"The index of '{input_word}' is: {index}")
else:
    print(f"'{input_word}' is not found in the IMDb dataset.")

The index of 'amazing' is: 477


In [ ]:
for doc_index in range(0,10):
  decoded_review = " ".join([reverse_word_index.get(i - 3, "?") for i in train_data[doc_index]])
  print(f'[{doc_index}]: {decoded_review}')

[0]: ? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't yo

In [ ]:
for doc_index in range(0,10):
  decoded_test = " ".join([reverse_word_index.get(i - 3, "?") for i in test_data[doc_index]])
  print(f'[{doc_index}]: {decoded_test}')

[0]: ? please give this one a miss br br ? ? and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allowed this one on his plate he almost seemed to know this wasn't going to work out and his performance was quite ? so all you madison fans give this a miss
[1]: ? this film requires a lot of patience because it focuses on mood and character development the plot is very simple and many of the scenes take place on the same set in frances ? the sandy dennis character apartment but the film builds to a disturbing climax br br the characters create an atmosphere ? with sexual tension and psychological ? it's very interesting that robert altman directed this considering the style and structure of his other films still the trademark altman audio style is evident here and there i think what really makes this film work is the brilliant performance by sandy dennis it's definitely one of her darker characters but she pl

# Performing Naive Bayes

In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

train_text = [" ".join([reverse_word_index.get(i - 3, "?") for i in doc]) for doc in train_data]
test_text = [" ".join([reverse_word_index.get(i - 3, "?") for i in doc]) for doc in test_data]

count_vectorizer = CountVectorizer()

pipeline = Pipeline([
    ('vect', count_vectorizer),
    ('clf', MultinomialNB())
])

param_grid = {
    'vect__max_features': [1000, 5000, 10000],
    'vect__ngram_range': [(1, 1), (1, 2)]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1)
grid_search.fit(train_text, train_labels)

print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

grid_search_preds = grid_search.best_estimator_.predict(test_text)

print("Grid Search + Naive Bayes Classifier:")
print("Accuracy:", accuracy_score(test_labels, grid_search_preds))
print("Classification Report:\n", classification_report(test_labels, grid_search_preds))
print("Confusion Matrix:\n", confusion_matrix(test_labels, grid_search_preds))

/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()
/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()


Best Parameters: {'vect__max_features': 10000, 'vect__ngram_range': (1, 2)}
Best Score: 0.8470000000000001
Grid Search + Naive Bayes Classifier:
Accuracy: 0.84368
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.86      0.85     12500
           1       0.85      0.83      0.84     12500

    accuracy                           0.84     25000
   macro avg       0.84      0.84      0.84     25000
weighted avg       0.84      0.84      0.84     25000

Confusion Matrix:
 [[10688  1812]
 [ 2096 10404]]


In [4]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from scipy.stats import randint

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
train_text = [" ".join([reverse_word_index.get(i - 3, "?") for i in doc]) for doc in train_data]
test_text = [" ".join([reverse_word_index.get(i - 3, "?") for i in doc]) for doc in test_data]
param_dist = {
    'vect__max_features': randint(1000, 10000),
    'vect__ngram_range': [(1, 1), (1, 2)]
}

random_search = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=10, cv=5, n_jobs=-1)
random_search.fit(train_text, train_labels)

print("Best Parameters (Random Search):", random_search.best_params_)
print("Best Score (Random Search):", random_search.best_score_)

random_search_preds = random_search.best_estimator_.predict(test_text)

print("Random Search + Naive Bayes Classifier:")
print("Accuracy:", accuracy_score(test_labels, random_search_preds))
print("Classification Report:\n", classification_report(test_labels, random_search_preds))
print("Confusion Matrix:\n", confusion_matrix(test_labels, random_search_preds))

Best Parameters (Random Search): {'vect__max_features': 7272, 'vect__ngram_range': (1, 2)}
Best Score (Random Search): 0.84092
Random Search + Naive Bayes Classifier:
Accuracy: 0.8398
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.85      0.84     12500
           1       0.85      0.83      0.84     12500

    accuracy                           0.84     25000
   macro avg       0.84      0.84      0.84     25000
weighted avg       0.84      0.84      0.84     25000

Confusion Matrix:
 [[10628  1872]
 [ 2133 10367]]


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

count_vectorizer = CountVectorizer()

train_text = [" ".join([reverse_word_index.get(i - 3, "?") for i in doc]) for doc in train_data]
test_text = [" ".join([reverse_word_index.get(i - 3, "?") for i in doc]) for doc in test_data]

nb_pipeline = Pipeline([
    ('vect', count_vectorizer),
    ('clf', MultinomialNB())
])
nb_pipeline.fit(train_text, train_labels)

nb_preds = nb_pipeline.predict(test_text)

print("Naive Bayes Classifier:")
print("Accuracy:", accuracy_score(test_labels, nb_preds))
print("Classification Report:\n", classification_report(test_labels, nb_preds))
print("Confusion Matrix:\n", confusion_matrix(test_labels, nb_preds))

Cross-Validation Scores: [0.832  0.8444 0.848  0.8322 0.8434]
Mean CV Accuracy: 0.8400000000000001

Naive Bayes Classifier:
Accuracy: 0.82368
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.87      0.83     12500
           1       0.86      0.78      0.82     12500

    accuracy                           0.82     25000
   macro avg       0.83      0.82      0.82     25000
weighted avg       0.83      0.82      0.82     25000

Confusion Matrix:
 [[10861  1639]
 [ 2769  9731]]


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

count_vectorizer = CountVectorizer()

train_text = [" ".join([reverse_word_index.get(i - 3, "?") for i in doc]) for doc in train_data]
test_text = [" ".join([reverse_word_index.get(i - 3, "?") for i in doc]) for doc in test_data]

alpha = 1.0

pipeline = Pipeline([
    ('vect', CountVectorizer()),
    ('clf', MultinomialNB(alpha=alpha))
])
nb_pipeline.fit(train_text, train_labels)

nb_preds = nb_pipeline.predict(test_text)

print("Naive Bayes Classifier:")
print("Accuracy:", accuracy_score(test_labels, nb_preds))
print("Classification Report:\n", classification_report(test_labels, nb_preds))
print("Confusion Matrix:\n", confusion_matrix(test_labels, nb_preds))


Naive Bayes Classifier:
Accuracy: 0.82368
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.87      0.83     12500
           1       0.86      0.78      0.82     12500

    accuracy                           0.82     25000
   macro avg       0.83      0.82      0.82     25000
weighted avg       0.83      0.82      0.82     25000

Confusion Matrix:
 [[10861  1639]
 [ 2769  9731]]


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

count_vectorizer = CountVectorizer()

train_text = count_vectorizer.fit_transform([" ".join([reverse_word_index.get(i - 3, "?") for i in doc]) for doc in train_data]).toarray()
test_text = count_vectorizer.transform([" ".join([reverse_word_index.get(i - 3, "?") for i in doc]) for doc in test_data]).toarray()

gnb_pipeline = Pipeline([
    ('clf', GaussianNB())
])
gnb_pipeline.fit(train_text, train_labels)
gnb_preds = gnb_pipeline.predict(test_text)

print("Gaussian Naive Bayes Classifier:")
print("Accuracy:", accuracy_score(test_labels, gnb_preds))
print("Classification Report:\n", classification_report(test_labels, gnb_preds))
print("Confusion Matrix:\n", confusion_matrix(test_labels, gnb_preds))

Gaussian Naive Bayes Classifier:
Accuracy: 0.66264
Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.86      0.72     12500
           1       0.77      0.47      0.58     12500

    accuracy                           0.66     25000
   macro avg       0.69      0.66      0.65     25000
weighted avg       0.69      0.66      0.65     25000

Confusion Matrix:
 [[10739  1761]
 [ 6673  5827]]


## Building your model

In [ ]:
from tensorflow import keras
from keras import layers
from keras import activations

model = keras.Sequential([
    layers.Dense(10, activation="relu", input_shape=(vocab_size,)),
    layers.Dense(1024, activation="relu"),  #First Hidden Layer
    layers.Dense(1024, activation="relu"),  #Second Hidden Layer

    layers.Dense(1, activation="sigmoid")
])

In [ ]:
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

In [ ]:
model.build()

In [ ]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_4 (Dense)             (None, 10)                100010    
                                                                 
 dense_5 (Dense)             (None, 1024)              11264     
                                                                 
 dense_6 (Dense)             (None, 1024)              1049600   
                                                                 
 dense_7 (Dense)             (None, 1)                 1025      
                                                                 
Total params: 1,161,899
Trainable params: 1,161,899
Non-trainable params: 0
_________________________________________________________________


In [ ]:
model.fit(x_train, y_train, epochs=4, batch_size=512)

Epoch 1/4
49/49 [==============================] - 6s 105ms/step - loss: 0.4644 - accuracy: 0.7706
Epoch 2/4
49/49 [==============================] - 4s 90ms/step - loss: 0.2556 - accuracy: 0.8997
Epoch 3/4
49/49 [==============================] - 5s 109ms/step - loss: 0.1986 - accuracy: 0.9256
Epoch 4/4
49/49 [==============================] - 4s 90ms/step - loss: 0.1576 - accuracy: 0.9393


In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"test_acc: {test_acc} \t test_loss: {test_loss}")

782/782 [==============================] - 4s 5ms/step - loss: 0.3184 - accuracy: 0.8766
test_acc: 0.8766000270843506 	 test_loss: 0.3183525502681732
